# Baseline: Killourhy & Maxion (2009) detectors

Reproduces the published EERs on 31 raw features (Euclidean 0.1706, Manhattan 0.1529,
Mahalanobis 0.1101, Scaled Manhattan 0.0962) and shows the effect of the 21 log features
(Scaled Manhattan | 21 log = the baseline used in `experiments.ipynb`). This notebook writes nothing to disk.

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))
import pandas as pd
from src import utils as U, models as M
from src.evaluation import ModelSpec, evaluate

df = U.load_data()
FEATS = {fs: U.make_features(df, fs)[0] for fs in ("raw31", "raw21", "log21")}
PUBLISHED = {"Euclidean": 0.1706, "Manhattan": 0.1529, "Mahalanobis": 0.1101, "Scaled Manhattan": 0.0962}

In [2]:
rows = []
for cls in (M.Euclidean, M.Manhattan, M.Mahalanobis, M.ScaledManhattan):
    row = {"detector": cls.name, "published (raw31)": PUBLISHED[cls.name]}
    for fs, X in FEATS.items():
        s = evaluate(X, ModelSpec(cls.name, cls), "test").summary()
        row[fs] = s["mean_eer"]
        if fs == "log21":
            row["log21 95% CI"] = f"[{s['ci95_low']:.4f}, {s['ci95_high']:.4f}]"
    rows.append(row)
table = pd.DataFrame(rows)
table.round(4)

,detector,published (raw31),raw31,raw21,log21,log21 95% CI
0,Euclidean,0.1706,0.1706,0.1674,0.1259,"[0.1083, 0.1439]"
1,Manhattan,0.1529,0.1529,0.1386,0.0998,"[0.0808, 0.1196]"
2,Mahalanobis,0.1101,0.1101,0.1101,0.0974,"[0.0825, 0.1127]"
3,Scaled Manhattan,0.0962,0.0962,0.0905,0.0855,"[0.0669, 0.1048]"
